# 🏆 Amazon ML Challenge 2026 — Business Entity Resolution

**Full pipeline:** Blocking → Feature Engineering → LightGBM → Threshold Tuning → Output

## Setup
1. Upload your dataset to Kaggle as a dataset named `amazon-ml-challenge-2026`
2. Make sure the dataset has the `student_resource/` folder structure
3. Enable Internet = OFF (no external API calls allowed)
4. Accelerator: CPU (P100 GPU optional for sentence-transformer re-ranker)
5. RAM: Standard (30GB should be enough)

**Dataset path on Kaggle:** `/kaggle/input/amazon-ml-challenge-2026/student_resource/`

## 📦 Install Dependencies

In [ ]:
!pip install -q lightgbm==4.4.0 rapidfuzz==3.9.0 jellyfish==1.0.3 datasketch==1.6.5 tqdm

## ⚙️ Configuration

In [ ]:
import os
import sys

# ─── Paths ────────────────────────────────────────────────────────────────────
# Kaggle path — update if your dataset name differs
KAGGLE_DATA_DIR = '/kaggle/input/amazon-ml-challenge-2026/student_resource'
LOCAL_DATA_DIR  = '.'   # local: path to student_resource/

DATA_DIR   = KAGGLE_DATA_DIR if os.path.exists(KAGGLE_DATA_DIR) else LOCAL_DATA_DIR
MODEL_DIR  = '/kaggle/working/models'
OUTPUT_DIR = '/kaggle/working/output'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Data dir:   {DATA_DIR}')
print(f'Model dir:  {MODEL_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

# ─── Hyperparameters ──────────────────────────────────────────────────────────
MAX_CANDIDATES  = 500   # max blocking candidates per S1 entity
NEG_PER_POS     = 3     # hard negatives per positive pair (training)
VAL_FRACTION    = 0.20  # fraction of S1 for validation
BATCH_SIZE      = 300_000  # pairs per feature-compute batch
RANDOM_SEED     = 42

LGBM_PARAMS = dict(
    objective='binary',
    metric='binary_logloss',
    boosting_type='gbdt',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=3,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=-1,
)

## 🛠️ Core Utilities

In [ ]:
"""Text normalization utilities."""
import re
import unicodedata
import collections
import csv
import os
import pickle
import json
import random
import numpy as np
from typing import Dict, List, Tuple, Set, Optional
from tqdm.auto import tqdm

# ─── Legal suffix dictionary ───────────────────────────────────────────────────
LEGAL_SUFFIXES = {
    'inc', 'incorporated', 'corp', 'corporation', 'co', 'company',
    'ltd', 'limited', 'llc', 'lp', 'llp', 'plc',
    'pvt', 'private', 'pte', 'grp', 'group', 'holdings', 'holding',
    'enterprises', 'enterprise', 'ventures', 'venture',
    'solutions', 'services', 'systems', 'international', 'intl', 'global',
    'worldwide', 'associates', 'association', 'foundation', 'trust',
    # French
    'sa', 'sarl', 'sas', 'sasu', 'sci', 'eurl', 'snc', 'sca', 'scs', 'ei', 'eirl',
    # India
    'pvtltd',
}

NAME_TOKEN_EXPAND = {
    '&': 'and', 'intl': 'international', 'natl': 'national',
    'mgmt': 'management', 'svcs': 'services', 'svc': 'service',
    'mfg': 'manufacturing', 'tech': 'technology', 'assoc': 'associates',
    'assn': 'association', 'univ': 'university', 'hosp': 'hospital',
    'med': 'medical', 'sys': 'systems', 'bldg': 'building',
    'ctr': 'center', 'bros': 'brothers', 'pvt': 'private',
    'corp': 'corporation', 'inc': 'incorporated', 'ltd': 'limited',
}

ADDR_TOKEN_EXPAND = {
    'st': 'street', 'rd': 'road', 'ave': 'avenue', 'blvd': 'boulevard',
    'dr': 'drive', 'ct': 'court', 'ln': 'lane', 'hwy': 'highway',
    'pkwy': 'parkway', 'n': 'north', 's': 'south', 'e': 'east', 'w': 'west',
    'apt': 'apartment', 'ste': 'suite', 'bldg': 'building',
}

_PUNCT_RE = re.compile(r'[^\w\s]', re.UNICODE)
_MULTI_SPACE_RE = re.compile(r'\s+')
_LEADING_NOISE_RE = re.compile(r'^[\-\*<>\.\s]+')
_US_ZIP_RE = re.compile(r'\b(\d{5})(?:-\d{4})?\b')
_IN_PIN_RE = re.compile(r'\b(\d{6})\b')
_FR_CP_RE  = re.compile(r'\b(\d{5})\b')
_GENERIC_NUM_RE = re.compile(r'\b(\d{4,6})\b')
_STREET_NUM_RE = re.compile(r'^\s*(\d+(?:\s*[A-Za-z])?(?:/\d+)?)\b')

def tokenize(text):
    if not text: return []
    text = unicodedata.normalize('NFC', str(text)).lower()
    text = _LEADING_NOISE_RE.sub('', text)
    text = _PUNCT_RE.sub(' ', text)
    return [t for t in _MULTI_SPACE_RE.sub(' ', text).strip().split() if t]

def normalize_name(name, expand=True):
    if not name: return ''
    toks = tokenize(name)
    if expand: toks = [NAME_TOKEN_EXPAND.get(t, t) for t in toks]
    return ' '.join(toks)

def core_name(name):
    toks = normalize_name(name, expand=True).split()
    filtered = [t for t in toks if t not in LEGAL_SUFFIXES]
    return ' '.join(filtered) if filtered else ' '.join(toks)

def normalize_address(addr, expand=True):
    if not addr: return ''
    toks = tokenize(addr)
    if expand: toks = [ADDR_TOKEN_EXPAND.get(t, t) for t in toks]
    return ' '.join(toks)

def extract_postal_code(address, country=''):
    if not address: return None
    country = (country or '').upper()
    if country == 'US':
        m = _US_ZIP_RE.search(address); return m.group(1) if m else None
    elif country in ('INDIA', 'IN'):
        m = _IN_PIN_RE.search(address); return m.group(1) if m else None
    elif country in ('FRANCE', 'FR'):
        m = _FR_CP_RE.search(address); return m.group(1) if m else None
    else:
        for p in [_IN_PIN_RE, _US_ZIP_RE, _GENERIC_NUM_RE]:
            m = p.search(address)
            if m: return m.group(1)
    return None

def extract_street_num(address):
    if not address: return None
    m = _STREET_NUM_RE.match(address.strip())
    return m.group(1).strip() if m else None

def name_trigrams(name):
    s = core_name(name).replace(' ', '')
    if len(s) < 3: return [s] if s else []
    return [s[i:i+3] for i in range(len(s)-2)]

def soundex(name):
    try:
        import jellyfish
        for t in core_name(name).split():
            if all(ord(c) < 128 for c in t) and len(t) >= 2:
                return jellyfish.soundex(t)
    except: pass
    return None

def nysiis(name):
    try:
        import jellyfish
        for t in core_name(name).split():
            if all(ord(c) < 128 for c in t) and len(t) >= 2:
                return jellyfish.nysiis(t)
    except: pass
    return None

def full_soundex(name):
    try:
        import jellyfish
        toks = [t for t in core_name(name).split()
                if all(ord(c)<128 for c in t) and len(t)>=2 and t not in LEGAL_SUFFIXES]
        return ''.join(jellyfish.soundex(t) for t in toks[:3]) if toks else ''
    except: return ''

print('✅ Text utilities defined')

In [ ]:
"""I/O helpers."""

def load_source(path):
    records = []
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            records.append({
                'entity_id': (row.get('entity_id') or '').strip(),
                'business_name': row.get('business_name') or '',
                'business_address': row.get('business_address') or '',
                'country': row.get('country') or '',
            })
    return records

def load_ground_truth(path):
    gt = {}
    with open(path, encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            s1_id = row['source1_entity_id'].strip()
            raw = row.get('matched_entity_ids') or ''
            gt[s1_id] = [m.strip() for m in raw.split(',') if m.strip()]
    return gt

def write_tsv(path, header, rows):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f, delimiter='\t', lineterminator='\n')
        w.writerow(header)
        for row in rows: w.writerow(row)

def write_matching_results(predictions, path, all_s1_ids):
    rows = []
    for s1_id in all_s1_ids:
        matched = predictions.get(s1_id, [])
        seen, deduped = set(), []
        for mid in matched:
            if mid not in seen: seen.add(mid); deduped.append(mid)
        rows.append([s1_id, ','.join(deduped)])
    write_tsv(path, ['source1_entity_id', 'matched_entity_ids'], rows)

def write_candidate_pairs(candidates, path, all_s1_ids):
    rows = []
    for s1_id in all_s1_ids:
        cands = candidates.get(s1_id, [])
        seen, deduped = set(), []
        for cid in cands:
            if cid not in seen: seen.add(cid); deduped.append(cid)
        rows.append([s1_id, ','.join(deduped)])
    write_tsv(path, ['source1_entity_id', 'candidate_entity_ids'], rows)

print('✅ I/O helpers defined')

In [ ]:
"""F0.5 metric."""

def f05_per_entity(pred, truth):
    p, t = set(pred), set(truth)
    if not p and not t: return 1.0
    if not p or not t: return 0.0
    tp = len(p & t)
    if tp == 0: return 0.0
    precision, recall = tp/len(p), tp/len(t)
    return 1.25 * precision * recall / (0.25 * precision + recall)

def f05_macro(pred_dict, truth_dict):
    scores = [f05_per_entity(pred_dict.get(k, []), v) for k, v in truth_dict.items()]
    return sum(scores)/len(scores) if scores else 0.0

def sweep_threshold(scores_by_entity, truth_dict, thresholds=None):
    if thresholds is None:
        thresholds = [round(i*0.05, 2) for i in range(2, 20)]
    best, best_t = 0.0, 0.5
    results = []
    for t in thresholds:
        pred = {k: [cid for cid, sc in v if sc >= t] for k, v in scores_by_entity.items()}
        f = f05_macro(pred, truth_dict)
        results.append({'threshold': t, 'f05': f})
        if f > best: best, best_t = f, t
    return {'best_threshold': best_t, 'best_f05': best, 'results': results}

print('✅ Metric helpers defined')

## 📥 Load Data

In [ ]:
print('Loading training data...')
s1_train = load_source(os.path.join(DATA_DIR, 'dataset/train/train_source1.tsv'))
s2_train = load_source(os.path.join(DATA_DIR, 'dataset/train/train_source2.tsv'))
s3_train = load_source(os.path.join(DATA_DIR, 'dataset/train/train_source3.tsv'))
gt_train = load_ground_truth(os.path.join(DATA_DIR, 'dataset/train/train_ground_truth.tsv'))

print(f'S1 train: {len(s1_train):,}')
print(f'S2 train: {len(s2_train):,}')
print(f'S3 train: {len(s3_train):,}')
print(f'GT rows:  {len(gt_train):,}')

singletons = sum(1 for v in gt_train.values() if not v)
print(f'Singletons: {singletons:,} ({singletons/len(gt_train)*100:.1f}%)')

print('\nLoading test data...')
s1_test = load_source(os.path.join(DATA_DIR, 'dataset/test/test_source1.tsv'))
s2_test = load_source(os.path.join(DATA_DIR, 'dataset/test/test_source2.tsv'))
s3_test = load_source(os.path.join(DATA_DIR, 'dataset/test/test_source3.tsv'))
print(f'S1 test: {len(s1_test):,}')
print(f'S2 test: {len(s2_test):,}')
print(f'S3 test: {len(s3_test):,}')

# Country distribution in test
test_countries = collections.Counter(r['country'] for r in s1_test)
print(f'\nTest S1 country distribution: {dict(test_countries)}')

## ✂️ Train / Validation Split

In [ ]:
rng = random.Random(RANDOM_SEED)

# Stratify by country
by_country = collections.defaultdict(list)
for row in s1_train:
    by_country[row['country']].append(row)

train_s1, val_s1 = [], []
for country, rows in by_country.items():
    rng.shuffle(rows)
    split = max(1, int(len(rows) * (1 - VAL_FRACTION)))
    train_s1.extend(rows[:split])
    val_s1.extend(rows[split:])

train_ids = {r['entity_id'] for r in train_s1}
val_ids   = {r['entity_id'] for r in val_s1}
train_gt  = {k: v for k, v in gt_train.items() if k in train_ids}
val_gt    = {k: v for k, v in gt_train.items() if k in val_ids}

print(f'Train S1: {len(train_s1):,} | Val S1: {len(val_s1):,}')
print(f'Country distribution:')
for c, rows in sorted(by_country.items()):
    n = len(rows)
    n_val = int(n * VAL_FRACTION)
    print(f'  {c}: {n:,} total → {n-n_val:,} train / {n_val:,} val')

## 🔍 Stage 1: Blocking (Candidate Generation)

In [ ]:
"""Multi-key inverted-index blocking."""

def blocking_keys(row):
    """Generate all blocking keys for a single record."""
    name    = row.get('business_name', '') or ''
    address = row.get('business_address', '') or ''
    country = (row.get('country', '') or '').strip()
    keys = []

    postal   = extract_postal_code(address, country)
    nprefix3 = core_name(name)[:3]
    nprefix5 = core_name(name)[:5]
    sdx      = soundex(name)
    nys      = nysiis(name)
    cn       = core_name(name)

    if postal and nprefix3:
        keys.append(f'BK1|{country}|{postal[:4]}|{nprefix3}')  # postal + name prefix
    if sdx:
        keys.append(f'BK2|{country}|{sdx}')                    # soundex
    if nys:
        keys.append(f'BK3|{country}|{nys}')                    # NYSIIS
    if cn and len(cn) >= 3:
        keys.append(f'BK5|{country}|{cn}')                     # exact core name
    if nprefix5:
        keys.append(f'BK6|{country}|{nprefix5}')               # name prefix-5
    if postal:
        keys.append(f'BK7|{country}|{postal}')                 # postal only

    # Street number + soundex of next word
    snum = extract_street_num(address)
    addr_toks = normalize_address(address).split()
    if snum and len(addr_toks) >= 2:
        try:
            import jellyfish
            next_tok = addr_toks[1] if addr_toks[0].isdigit() else addr_toks[0]
            if all(ord(c)<128 for c in next_tok) and len(next_tok)>=2:
                asdx = jellyfish.soundex(next_tok)
                keys.append(f'BK4|{country}|{snum}|{asdx}')
        except: pass

    return keys


def build_index(records, verbose=True):
    """Build inverted index: blocking_key → [entity_ids]."""
    idx = collections.defaultdict(list)
    for row in tqdm(records, desc='Building index', disable=not verbose):
        for key in blocking_keys(row):
            idx[key].append(row['entity_id'])
    return dict(idx)


def build_ngram_index(records, verbose=True):
    """Build name trigram index for supplemental recall."""
    idx = collections.defaultdict(set)
    for row in tqdm(records, desc='Building n-gram index', disable=not verbose):
        for gram in name_trigrams(row.get('business_name', '') or ''):
            idx[gram].add(row['entity_id'])
    return dict(idx)


def get_candidates(s1_records, index, ngram_idx=None, ngram_min_overlap=3,
                   max_cands=MAX_CANDIDATES, verbose=True):
    """For each S1 entity, retrieve all blocking candidates from S2+S3."""
    candidates = {}
    for row in tqdm(s1_records, desc='Getting candidates', disable=not verbose):
        s1_id = row['entity_id']
        cand_set = set()

        # Key-based blocking
        for key in blocking_keys(row):
            for mid in index.get(key, []):
                if mid.startswith('S2-') or mid.startswith('S3-'):
                    cand_set.add(mid)

        # N-gram supplemental
        if ngram_idx:
            overlap = collections.Counter()
            for gram in set(name_trigrams(row.get('business_name','') or '')):
                for eid in ngram_idx.get(gram, set()):
                    overlap[eid] += 1
            for eid, cnt in overlap.items():
                if cnt >= ngram_min_overlap and (eid.startswith('S2-') or eid.startswith('S3-')):
                    cand_set.add(eid)

        cand_list = list(cand_set)
        if len(cand_list) > max_cands:
            cand_list = sorted(cand_list)[:max_cands]  # deterministic cap
        candidates[s1_id] = cand_list

    return candidates


def eval_blocking(candidates, gt):
    """Measure blocking recall."""
    total, found, total_cands = 0, 0, 0
    for s1_id, true_m in gt.items():
        cands = set(candidates.get(s1_id, []))
        total += len(true_m)
        found += sum(1 for m in true_m if m in cands)
        total_cands += len(cands)
    n = len(gt)
    return {
        'recall': found/total if total else 0.0,
        'found': found, 'total': total,
        'avg_cands': total_cands/n if n else 0.0,
    }

print('✅ Blocking functions defined')

In [ ]:
# ─── Run blocking ──────────────────────────────────────────────────────────────
s2s3_train = s2_train + s3_train
s2s3_test  = s2_test  + s3_test

print('=== Building S2+S3 indexes (train) ===')
idx_train      = build_index(s2s3_train)
ngram_idx_train = build_ngram_index(s2s3_train)

print('\n=== Generating candidates (train split) ===')
cands_train = get_candidates(train_s1, idx_train, ngram_idx_train)
stats = eval_blocking(cands_train, train_gt)
print(f"Train blocking recall: {stats['recall']:.4f} | Avg cands/entity: {stats['avg_cands']:.1f}")

print('\n=== Generating candidates (val split) ===')
cands_val = get_candidates(val_s1, idx_train, ngram_idx_train)
stats_val = eval_blocking(cands_val, val_gt)
print(f"Val blocking recall: {stats_val['recall']:.4f} | Avg cands/entity: {stats_val['avg_cands']:.1f}")

# IMPORTANT: if blocking recall < 0.95, increase MAX_CANDIDATES or add more blocking keys
if stats_val['recall'] < 0.95:
    print('⚠️  WARNING: Blocking recall is below 0.95! Consider adding more blocking keys.')

## 🔧 Stage 2: Feature Engineering

In [ ]:
"""Feature computation."""
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

FEATURE_NAMES = [
    'name_wratio', 'name_ratio', 'name_token_sort', 'name_partial',
    'name_jaccard_tok', 'name_jaccard_tri', 'name_tfidf_cos',
    'core_exact', 'core_wratio', 'core_tri_jac',
    'addr_jaccard_tok', 'addr_ratio', 'addr_tfidf_cos',
    'postal_exact', 'postal_prefix', 'street_num_exact', 'city_overlap',
    'soundex_match', 'nysiis_match', 'full_sdx_match',
    'country_match', 'is_s2', 'name_addr_product',
    'name_comp_s1', 'name_comp_cand', 'addr_comp_s1', 'addr_comp_cand',
    'name_has_digits',
]
N_FEAT = len(FEATURE_NAMES)

def jaccard(a, b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    sa, sb = set(a), set(b)
    return len(sa & sb) / len(sa | sb)

def jaccard_toks(ta, tb): return jaccard(ta.split(), tb.split())
def jaccard_tris(na, nb): return jaccard(name_trigrams(na), name_trigrams(nb))

def tok_overlap(a, b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    sa, sb = set(a), set(b)
    return len(sa & sb) / min(len(sa), len(sb))

def compute_pair(s1, cand, nv=None, av=None):
    """Compute feature vector for one pair. nv/av are TF-IDF vectorizers (optional)."""
    feat = np.zeros(N_FEAT, dtype=np.float32)

    s1n  = normalize_name(s1.get('business_name','') or '')
    cn1  = core_name(s1.get('business_name','') or '')
    s1a  = normalize_address(s1.get('business_address','') or '')
    s1c  = (s1.get('country','') or '').strip()

    cn2  = normalize_name(cand.get('business_name','') or '')
    cnc  = core_name(cand.get('business_name','') or '')
    ca   = normalize_address(cand.get('business_address','') or '')
    cc   = (cand.get('country','') or '').strip()
    cid  = cand.get('entity_id','')

    # Name
    if s1n and cn2:
        feat[0]  = fuzz.WRatio(s1n, cn2) / 100
        feat[1]  = fuzz.ratio(s1n, cn2) / 100
        feat[2]  = fuzz.token_sort_ratio(s1n, cn2) / 100
        feat[3]  = fuzz.partial_ratio(s1n, cn2) / 100
        feat[4]  = jaccard_toks(s1n, cn2)
        feat[5]  = jaccard_tris(s1n, cn2)
    if cn1 and cnc:
        feat[7]  = float(cn1 == cnc)
        feat[8]  = fuzz.WRatio(cn1, cnc) / 100
        feat[9]  = jaccard_tris(cn1, cnc)

    # Address
    if s1a and ca:
        feat[10] = jaccard_toks(s1a, ca)
        feat[11] = fuzz.ratio(s1a, ca) / 100
    elif not s1a and not ca:
        feat[10] = feat[11] = 1.0

    s1_raw = s1.get('business_address','') or ''
    ca_raw = cand.get('business_address','') or ''
    p1 = extract_postal_code(s1_raw, s1c)
    p2 = extract_postal_code(ca_raw, cc)
    if p1 and p2:
        feat[13] = float(p1 == p2)
        feat[14] = float(p1[:3] == p2[:3])
    n1 = extract_street_num(s1_raw)
    n2 = extract_street_num(ca_raw)
    if n1 and n2: feat[15] = float(n1 == n2)
    at1 = s1a.split()
    at2 = ca.split()
    if at1 and at2: feat[16] = tok_overlap(at1[-3:], at2[-3:])

    # Phonetic
    s1n_raw = s1.get('business_name','') or ''
    cn_raw  = cand.get('business_name','') or ''
    sd1, sd2 = soundex(s1n_raw), soundex(cn_raw)
    feat[17] = float(sd1 and sd2 and sd1==sd2)
    ny1, ny2 = nysiis(s1n_raw), nysiis(cn_raw)
    feat[18] = float(ny1 and ny2 and ny1==ny2)
    fs1, fs2 = full_soundex(s1n_raw), full_soundex(cn_raw)
    feat[19] = float(fs1 and fs2 and fs1==fs2)

    # Meta
    feat[20] = float(s1c.upper() == cc.upper())
    feat[21] = float(cid.startswith('S2-'))
    feat[22] = feat[0] * feat[10]
    feat[23] = float(len(cn1.split()))
    feat[24] = float(len(cnc.split()))
    feat[25] = float(len(s1a.split()))
    feat[26] = float(len(ca.split()))
    feat[27] = float(bool(re.search(r'\d', s1n_raw + cn_raw)))

    return feat

print('✅ Feature functions defined')

In [ ]:
# ─── Fit TF-IDF vectorizers on ALL records (train + s2 + s3) ──────────────────
print('Fitting TF-IDF vectorizers...')
all_records = s1_train + s2_train + s3_train

name_vec = TfidfVectorizer(
    max_features=50_000, ngram_range=(1,2),
    analyzer='char_wb', min_df=2, sublinear_tf=True,
)
addr_vec = TfidfVectorizer(
    max_features=50_000, ngram_range=(1,2),
    analyzer='word', min_df=2, sublinear_tf=True,
)
all_names  = [normalize_name(r.get('business_name','') or '') for r in all_records]
all_addrs  = [normalize_address(r.get('business_address','') or '') for r in all_records]
name_vec.fit(all_names)
addr_vec.fit(all_addrs)
print(f'Name vocab: {len(name_vec.vocabulary_):,} | Addr vocab: {len(addr_vec.vocabulary_):,}')

In [ ]:
def compute_features_batch(pairs, nv=None, av=None):
    """Compute feature matrix for a list of (s1_row, cand_row) pairs."""
    X = np.array([compute_pair(s, c) for s, c in pairs], dtype=np.float32)

    if nv is not None and len(pairs) > 0:
        s1_names   = [normalize_name(p[0].get('business_name','') or '') for p in pairs]
        cand_names = [normalize_name(p[1].get('business_name','') or '') for p in pairs]
        sv = nv.transform(s1_names)
        cv = nv.transform(cand_names)
        # Efficient row-wise cosine similarity
        sv_norm = sv.multiply(1 / (np.sqrt(sv.power(2).sum(axis=1)) + 1e-9))
        cv_norm = cv.multiply(1 / (np.sqrt(cv.power(2).sum(axis=1)) + 1e-9))
        cos = np.array(sv_norm.multiply(cv_norm).sum(axis=1)).flatten()
        X[:, 6] = cos.astype(np.float32)

    if av is not None and len(pairs) > 0:
        s1_addrs   = [normalize_address(p[0].get('business_address','') or '') for p in pairs]
        cand_addrs = [normalize_address(p[1].get('business_address','') or '') for p in pairs]
        sv = av.transform(s1_addrs)
        cv = av.transform(cand_addrs)
        sv_norm = sv.multiply(1 / (np.sqrt(sv.power(2).sum(axis=1)) + 1e-9))
        cv_norm = cv.multiply(1 / (np.sqrt(cv.power(2).sum(axis=1)) + 1e-9))
        cos = np.array(sv_norm.multiply(cv_norm).sum(axis=1)).flatten()
        X[:, 12] = cos.astype(np.float32)

    return X

print('✅ Batch feature computation defined')

## 🧱 Build Training / Validation Pairs

In [ ]:
s23_map = {r['entity_id']: r for r in s2_train + s3_train}
rng2 = random.Random(RANDOM_SEED + 1)

# ─── Training pairs ────────────────────────────────────────────────────────────
print('Building training pairs...')
train_pairs, train_labels = [], []
for s1_row in tqdm(train_s1, desc='Train pairs'):
    s1_id = s1_row['entity_id']
    true_m = set(train_gt.get(s1_id, []))
    cand_pool = cands_train.get(s1_id, [])

    # Positive: all true matches (from GT, not just blocking)
    for cid in true_m:
        cand_row = s23_map.get(cid)
        if cand_row:
            train_pairs.append((s1_row, cand_row))
            train_labels.append(1)

    # Negatives: hard negatives from blocking pool
    neg_pool = [c for c in cand_pool if c not in true_m]
    n_neg = min(len(neg_pool), NEG_PER_POS * max(1, len(true_m)))
    for cid in (rng2.sample(neg_pool, n_neg) if len(neg_pool) > n_neg else neg_pool):
        cand_row = s23_map.get(cid)
        if cand_row:
            train_pairs.append((s1_row, cand_row))
            train_labels.append(0)

y_train = np.array(train_labels, dtype=np.int32)
print(f'Train pairs: {len(train_pairs):,} | Pos: {y_train.sum():,} | Neg: {(~y_train.astype(bool)).sum():,}')

In [ ]:
# ─── Compute training features ─────────────────────────────────────────────────
print('Computing training features...')
X_train_list = []
for start in tqdm(range(0, len(train_pairs), BATCH_SIZE), desc='Train feature batches'):
    batch = train_pairs[start:start+BATCH_SIZE]
    X_train_list.append(compute_features_batch(batch, name_vec, addr_vec))
X_train = np.vstack(X_train_list)
del X_train_list
print(f'X_train shape: {X_train.shape}')

In [ ]:
# ─── Validation pairs (ALL blocking candidates, for accurate F0.5 sweep) ───────
print('Building validation pairs...')
val_pairs, val_labels, val_meta = [], [], []
for s1_row in tqdm(val_s1, desc='Val pairs'):
    s1_id = s1_row['entity_id']
    true_m = set(val_gt.get(s1_id, []))
    for cid in cands_val.get(s1_id, []):
        cand_row = s23_map.get(cid)
        if cand_row:
            val_pairs.append((s1_row, cand_row))
            val_labels.append(1 if cid in true_m else 0)
            val_meta.append((s1_id, cid))

y_val = np.array(val_labels, dtype=np.int32)
print(f'Val pairs: {len(val_pairs):,} | Pos: {y_val.sum():,}')

print('Computing validation features...')
X_val_list = []
for start in tqdm(range(0, len(val_pairs), BATCH_SIZE), desc='Val feature batches'):
    batch = val_pairs[start:start+BATCH_SIZE]
    X_val_list.append(compute_features_batch(batch, name_vec, addr_vec))
X_val = np.vstack(X_val_list)
del X_val_list
print(f'X_val shape: {X_val.shape}')

## 🌟 Stage 3: Train LightGBM

In [ ]:
import lightgbm as lgb

print('Training LightGBM...')
params = LGBM_PARAMS.copy()
n_estimators = params.pop('n_estimators', 500)
lr = params.pop('learning_rate', 0.05)

model = lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=lr, **params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(50)],
)
print(f'Best iteration: {model.best_iteration_}')

# Feature importance
importance = sorted(zip(FEATURE_NAMES, model.feature_importances_), key=lambda x: -x[1])
print('\nTop 15 features:')
for name, imp in importance[:15]:
    bar = '█' * int(imp / max(v for _, v in importance) * 20)
    print(f'  {name:<25} {imp:5.0f}  {bar}')

## 🎯 Stage 4: Threshold Tuning

In [ ]:
print('Scoring validation pairs...')
val_scores = model.predict_proba(X_val)[:, 1]

# Aggregate per S1 entity
scores_by_entity = {}
for (s1_id, cid), score in zip(val_meta, val_scores):
    scores_by_entity.setdefault(s1_id, []).append((cid, float(score)))

# Ensure all val entities have an entry (even empty → singleton)
for s1_row in val_s1:
    s1_id = s1_row['entity_id']
    if s1_id not in scores_by_entity:
        scores_by_entity[s1_id] = []

# Sweep
print('Sweeping threshold...')
sweep = sweep_threshold(scores_by_entity, val_gt)
BEST_THRESHOLD = sweep['best_threshold']
BEST_F05       = sweep['best_f05']

print(f'\nBest threshold: {BEST_THRESHOLD:.2f}  →  Val F₀.₅ = {BEST_F05:.4f}')
print('\nFull sweep:')
for r in sweep['results']:
    marker = ' ← BEST' if r['threshold'] == BEST_THRESHOLD else ''
    print(f"  thresh={r['threshold']:.2f}  F₀.₅={r['f05']:.4f}{marker}")

## 💾 Save Model Artifacts

In [ ]:
print('Saving artifacts...')
with open(os.path.join(MODEL_DIR, 'lgbm_model.pkl'), 'wb') as f: pickle.dump(model, f)
with open(os.path.join(MODEL_DIR, 'tfidf_name.pkl'), 'wb') as f: pickle.dump(name_vec, f)
with open(os.path.join(MODEL_DIR, 'tfidf_addr.pkl'), 'wb') as f: pickle.dump(addr_vec, f)

training_results = {
    'best_threshold': BEST_THRESHOLD,
    'val_f05': BEST_F05,
    'blocking_recall_train': stats['recall'],
    'blocking_recall_val': stats_val['recall'],
    'n_train_pairs': len(train_pairs),
    'n_val_pairs': len(val_pairs),
    'best_lgbm_iteration': model.best_iteration_,
    'feature_importance': {n: int(i) for n, i in importance},
}
with open(os.path.join(MODEL_DIR, 'training_config.json'), 'w') as f:
    json.dump(training_results, f, indent=2)

print(f'✅ Model saved to {MODEL_DIR}')
print(f'   lgbm_model.pkl, tfidf_name.pkl, tfidf_addr.pkl, training_config.json')

## 🔮 Stage 5: Test Inference

In [ ]:
print('=== Blocking (test) ===')
s2s3_test_all = s2_test + s3_test
idx_test      = build_index(s2s3_test_all)
ngram_idx_test = build_ngram_index(s2s3_test_all)
cands_test    = get_candidates(s1_test, idx_test, ngram_idx_test)
total_test_cands = sum(len(v) for v in cands_test.values())
print(f'Total test candidate pairs: {total_test_cands:,}')
print(f'Avg per entity: {total_test_cands / max(1, len(s1_test)):.1f}')

In [ ]:
print('=== Scoring test pairs ===')
s23_test_map = {r['entity_id']: r for r in s2s3_test_all}
test_pairs, test_meta = [], []

for s1_row in s1_test:
    s1_id = s1_row['entity_id']
    for cid in cands_test.get(s1_id, []):
        cand_row = s23_test_map.get(cid)
        if cand_row:
            test_pairs.append((s1_row, cand_row))
            test_meta.append((s1_id, cid))

print(f'Test pairs to score: {len(test_pairs):,}')

all_scores = []
for start in tqdm(range(0, len(test_pairs), BATCH_SIZE), desc='Scoring batches'):
    batch = test_pairs[start:start+BATCH_SIZE]
    X_batch = compute_features_batch(batch, name_vec, addr_vec)
    scores = model.predict_proba(X_batch)[:, 1]
    all_scores.extend(scores.tolist())

print(f'Scored {len(all_scores):,} pairs')

In [ ]:
# ─── Apply threshold ──────────────────────────────────────────────────────────
all_s1_ids = [r['entity_id'] for r in s1_test]
predictions = {s1_id: [] for s1_id in all_s1_ids}

for (s1_id, cid), score in zip(test_meta, all_scores):
    if score >= BEST_THRESHOLD:
        predictions[s1_id].append(cid)

n_with_match = sum(1 for v in predictions.values() if v)
n_singletons = len(all_s1_ids) - n_with_match
n_total_preds = sum(len(v) for v in predictions.values())

print(f'Threshold: {BEST_THRESHOLD:.2f}')
print(f'Entities with ≥1 match: {n_with_match:,}')
print(f'Singletons (no match):  {n_singletons:,}')
print(f'Total match predictions: {n_total_preds:,}')

## 📤 Write Output Files

In [ ]:
matching_path  = os.path.join(OUTPUT_DIR, 'matching_results.tsv')
candidate_path = os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv')

write_matching_results(predictions, matching_path, all_s1_ids)
write_candidate_pairs(cands_test, candidate_path, all_s1_ids)

print(f'✅ matching_results.tsv  → {matching_path}')
print(f'✅ candidate_pairs.tsv  → {candidate_path}')

# Sanity check: peek at first 5 rows
print('\n--- matching_results.tsv (first 5 rows) ---')
with open(matching_path) as f:
    for i, line in enumerate(f):
        if i >= 6: break
        print(repr(line.rstrip()))

## ✅ Validate Submission

In [ ]:
import subprocess
validator = os.path.join(DATA_DIR, 'utils', 'validate_submission.py')
if os.path.exists(validator):
    result = subprocess.run(
        [sys.executable, validator,
         '--matching', matching_path,
         '--candidate', candidate_path,
         '--test-dir', os.path.join(DATA_DIR, 'dataset/test')],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('⚠️ VALIDATION FAILED:')
        print(result.stderr)
    else:
        print('🎉 PASS — safe to submit!')
else:
    print(f'Validator not found at {validator}')

## 📊 Error Analysis (optional but recommended)
Run this cell to understand where the model is failing.

In [ ]:
# ─── Per-country F0.5 breakdown on validation set ─────────────────────────────
val_preds_at_best = {}
for s1_row in val_s1:
    s1_id = s1_row['entity_id']
    val_preds_at_best[s1_id] = [
        cid for cid, sc in scores_by_entity.get(s1_id, []) if sc >= BEST_THRESHOLD
    ]

country_scores = collections.defaultdict(list)
for s1_row in val_s1:
    s1_id = s1_row['entity_id']
    country = s1_row.get('country', 'unknown')
    score = f05_per_entity(val_preds_at_best.get(s1_id, []), val_gt.get(s1_id, []))
    country_scores[country].append(score)

print('=== Per-country F₀.₅ on validation set ===')
for country, scores in sorted(country_scores.items()):
    print(f'{country:10s}: {sum(scores)/len(scores):.4f} ({len(scores):,} entities)')

# ─── False positive / false negative examples ─────────────────────────────────
print('\n=== False Positive Examples (wrong merges) ===')
fps = []
for s1_row in val_s1[:100]:  # sample first 100
    s1_id = s1_row['entity_id']
    pred = set(val_preds_at_best.get(s1_id, []))
    truth = set(val_gt.get(s1_id, []))
    for cid in (pred - truth):
        fps.append((s1_id, cid, s1_row))
        if len(fps) >= 3: break
    if len(fps) >= 3: break

for s1_id, cid, s1_row in fps:
    cand = s23_map.get(cid, {})
    print(f"  S1: {s1_row['business_name']!r} @ {s1_row['business_address']!r}")
    print(f"  FP: {cand.get('business_name','')!r} @ {cand.get('business_address','')!r}")
    print()

## 🎉 Done!

Your output files are at:
- `output/matching_results.tsv` — **upload this to the leaderboard**
- `output/candidate_pairs.tsv` — include in the final submission zip

**Next steps:**
1. Upload `matching_results.tsv` to the portal for a leaderboard score
2. Analyze the error analysis output above to find improvement areas
3. Iterate: adjust threshold, add features, tweak blocking keys
4. For the final submission zip, include both TSVs + source code + README